# Develop

## Import

In [ ]:
import numpy as np
import pandas as pd
import json

import pyarrow as pa
import pyarrow.parquet as pq

from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, row_number, lit

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import average_precision_score

from typing import List

## Model learning

In [ ]:
def train_in_chunks(files_list: List[str] = None,
                    cols_json_file: str = None,
                    batchsize: int = 20000,
                    model: CatBoostClassifier = None,
                    auto_nan_filler: bool = True):
    """
    Функция для последовательного обучения модели по чанкам. Принимает следующие аргументы:
    files_list - список с файлами для обучения модели (полные пути до файлов)
    cols_json_file - путь до json файла с инфой по колонкам
    batchsize - размер чанка (кол-во строк)
    model - модель
    """
    if files_list is None:
        raise AttributeError("Надо указать files_list - файл(ы) для обучения. Обязательно в списке")
    if cols_json_file is None:
        raise AttributeError("Надо указать cols_json_file - json файл с инфой по колонкам")
    if model is None:
        raise AttributeError("Надо указать model - действующую модель")
    
    with open(cols_json_file, 'r') as file:
        cols = json.load(file)
    
    feature_cols = cols['all']
    cat_cols = cols['cat']
    target_col = cols['target']
    
    for file in files_list:
        current_file = pq.ParquetFile(file)
        total_rows = current_file.metadata.num_rows
        total_batches = (total_rows + batchsize - 1) // batchsize
        print(f'В файле {file} {total_rows} строк. Это {total_batches} батчей')
        
        for batch_num, batch in enumerate(current_file.iter_batches(batch_size=batchsize)):
            print(f'Старт обработки {batch_num + 1}/{total_batches} батча')
            chunk = batch.to_pandas()
            
            print('В target батча содержится:')
            for value in chunk[target_col].unique():
                print(f'{value} - {len(chunk[chunk[target_col] == value])} штук')
            
            if auto_nan_filler:
                for col in cat_cols:
                    if col in chunk.columns:
                        chunk[col] = chunk[col].fillna('NaN').astype(str)
            
            X_batch = chunk[feature_cols]
            y_batch = chunk[target_col]
            train_pool = Pool(X_batch, y_batch, cat_features=cat_cols)
            
            print(f'Старт обучения {batch_num + 1}/{total_batches} батча')
            if batch_num == 0:
                model.fit(train_pool)
            else:
                model.fit(train_pool, init_model=model)
            print(f'Обучение {batch_num + 1}/{total_batches} батча завершено')
            print('==========================================================')

    model.save_model('../models/catboost_model.cbm')
    print(f'Обучение полностью завершено. Модель сохранена')

In [ ]:
valid_train_data_path = '../datasets/joined/train_data.parquet'

catboost_model = CatBoostClassifier(verbose=0)

train_in_chunks(files_list=[valid_train_data_path],
                cols_json_file='../datasets/joined/columns_list.json',
                model=catboost_model
                )

## Prediction

In [ ]:
def save_file_results(id_events: pd.Series,
                      predictions: np.ndarray,
                      save_path: str,
                      chunk_num: int):
    """
    Сохранение предсказаний вместе с идентификаторами в Parquet файл
    Принимает:
    id_events - объект pd.Series, который формируется в функции predict_in_chunks.
        Содержит данные об id операций. Нужен для коммита
    predictions - массив нампая с предсказаниями
    save_path - полный путь сохранения parquet файла
    chunk_num - флаговая переменная для указания номера чанка.
        Если чанк под номером 0 (первый), то надо создать новую таблицу. Иначе выполнить соединение с существующей
    """
    result_df = pd.DataFrame({
        'event_id': id_events,
        'predict': predictions
    })
    
    # Конвертируем в Arrow Table
    table = pa.Table.from_pandas(result_df)
    
    if chunk_num == 0:
        # Первый чанк или файл не существует - создаем новый
        pq.write_table(
            table,
            save_path,
            compression='snappy',
            flavor='spark'  # для совместимости со Spark если нужно
        )
        print(f"Создан файл: {save_path} ({len(result_df)} записей)")
    else:
        # Добавляем данные в существующий файл
        with pq.ParquetWriter(
            save_path,
            table.schema,
            compression='snappy',
            flavor='spark',
            append=True  # Важно: append режим
        ) as writer:
            writer.write_table(table)
        
        print(f"Добавлено {len(result_df)} записей в {save_path}")

In [ ]:
def predict_in_chunks(file: str = None,
                      batchsize: int = 20000,
                      cols_json_file: str = None,
                      return_type: str = 'class',  # 'proba' или 'class'
                      model: CatBoostClassifier = None,
                      save_path: List[str] = None
                      ):
    """
    Функция для предсказания на больших файлах по чанкам. Принимает:
    file - файл предсказания (полный путь)
    batchsize - размер чанка
    cols_json_file - путь до json файла с инфой по колонкам
    return_type - 'proba' для вероятностей, 'class' для классов
    model - используемая для предсказания модель
    save_path - полный путь для сохранения
    """
    
    if file is None:
        raise AttributeError("Надо указать file - файл для предсказания")
    if cols_json_file is None:
        raise AttributeError("Надо указать cols_json_file - json файл с инфой по колонкам")
    if model is None:
        raise AttributeError("Надо указать model - действующую модель")
    if save_path is None:
        raise AttributeError("Надо указать путь для сохранения итогового результата")
    
    with open(cols_json_file, 'r') as f:
        cols = json.load(f)
    
    feature_cols = cols['all']
    id_col = cols['id']
    
    total_rows = 0

    print(f"Обработка файла: {file}")
    
    current_file = pq.ParquetFile(file)
    total_rows = current_file.metadata.num_rows
    total_batches = (total_rows + batchsize - 1) // batchsize
    print(f'В файле {file} {total_rows} строк. Это {total_batches} батчей')
    
    for batch_idx, batch in enumerate(current_file.iter_batches(batch_size=batchsize)):
        print(f'Старт обработки {batch_idx + 1}/{total_batches} батча')
        df_chunk = batch.to_pandas()
        chunk = df_chunk[feature_cols]
        id_events=df_chunk[id_col]
        
        if return_type == 'proba':
            preds = model.predict_proba(chunk)[:, 1]
        else:
            preds = model.predict(chunk)
        
        save_file_results(id_events, preds, save_path, chunk_num=batch_idx)
        print(f'Обработка {batch_idx + 1}/{total_batches} батча завершена')

    print(f"Результаты сохранены в {save_path}")

In [ ]:
valid_predict_data_path = '../datasets/joined/valid_data.parquet'
valid_predict_saving_path = '../datasets/validation/validation_result.parquet'

predict_in_chunks(file=valid_predict_data_path,
                  cols_json_file='../datasets/joined/columns_list.json',
                  model=catboost_model,
                  save_path=valid_predict_saving_path)

## Error Analysis

In [ ]:
# Кастомная реализация sklearn.average_precision_score для big data
def average_precision_score(predict_path: str,
                            answers_path: str,
                            predict_column: str = 'target',
                            answers_column: str = 'target',
                            id_column: str = 'event_id'
                            ) -> float:
    """
    Расчет Average Precision (AP) метрики на больших данных с использованием Spark.
    
    AP = Σ (R_n - R_{n-1}) * P_n, где:
    - P_n - precision на n-ом пороге
    - R_n - recall на n-ом пороге
    
    predict_path: путь до parquet файла с предсказаниями (вероятности от 0 до 1)
    answers_path: путь до parquet файла с правильными ответами (бинарные метки 0/1)
    predict_column: название колонки с предсказаниями
    answers_column: название колонки с правильными ответами
    id_column: название колонки с идентификатором события
    
    Возвращает float: значение Average Precision метрики
    """
    
    spark = SparkSession.builder \
        .appName("AveragePrecisionCalculatorOptimized") \
        .config("spark.sql.adaptive.enabled", "true") \
        .getOrCreate()

    try:
        # Загружаем и объединяем данные
        predictions = spark.read.parquet(predict_path)
        answers = spark.read.parquet(answers_path)
        
        predict_cols = predictions.columns
        answers_cols = answers.columns
        if id_column not in predict_cols:
            raise ValueError(f"В файле предсказаний отсутствует колонка: {id_column}")
        if id_column not in answers_cols:
            raise ValueError(f"В файле ответов отсутствует колонка: {id_column}")
        if predict_column not in predict_cols:
            raise ValueError(f"В файле предсказаний отсутствует колонка: {predict_column}")
        if answers_column not in answers_cols:
            raise ValueError(f"В файле ответов отсутствует колонка: {answers_column}")
        
        
        merged = (predictions
                    .select(id_column, predict_column)
                    .join(answers.select(id_column, answers_column),
                        on=id_column,
                        how="inner")
                    .cache())
        
        total_count = merged.count()
        total_positives = merged.filter(col(answers_column) == 1).count()
        
        if total_positives == 0:
            return 0.0
        
        # Определяем окно для сортировки по убыванию предсказаний
        window_spec = Window.orderBy(col(predict_column).desc())
        
        # Добавляем индикаторы и кумулятивные суммы
        ap_data = merged.withColumn(
            "is_positive", col(answers_column)
        ).withColumn(
            "cumulative_tp", sum("is_positive").over(window_spec)
        ).withColumn(
            "row_num", row_number().over(window_spec)
        )
        
        # Расчет precision и recall для каждой строки
        # Используем lag для получения предыдущих значений
        result = ap_data.withColumn(
            "precision", col("cumulative_tp") / col("row_num")
        ).withColumn(
            "prev_tp", lag("cumulative_tp", 1, 0).over(window_spec)
        ).withColumn(
            "recall_diff", 
            (col("cumulative_tp") - col("prev_tp")) / lit(total_positives)
        ).agg(
            sum(col("recall_diff") * col("precision")).alias("average_precision")
        ).collect()[0][0]
        
        merged.unpersist()
        
        return float(result if result is not None else 0.0)
        
    finally:
        spark.stop()

In [ ]:
def target_class_mistakes_finder(predict_path: str,
                                 answers_path: str,
                                 predict_column: str = 'target',
                                 answers_column: str = 'target',
                                 id_column: str = 'event_id', 
                                 target_class: int = 1):
    """
    Функция по поиску расхождений предикта с известными данными
    именно с целевым классом - False Negative результаты
    Принимает:
    predict_path - полный путь к parquet файлу предсказания
    answers_path - полный путь к parquet файлу с ответами
    predcit_column - название колонки с лейблами предсказаний
    answers_column - название колонки с лейблами ответов
    id_column - название колонки для вывода
    target_class - фильтр по классу
    """
    # Создаем Spark сессию
    spark = SparkSession.builder \
        .appName("FindFalseNegatives") \
        .config("spark.sql.adaptive.enabled", "true") \
        .getOrCreate()

    try:
        # Загружаем данные и проверяем колонки
        predictions = spark.read.parquet(predict_path)
        answers = spark.read.parquet(answers_path)
        
        predict_cols = predictions.columns
        answers_cols = answers.columns
        
        if id_column not in predict_cols:
            raise ValueError(f"В файле предсказаний отсутствует колонка: {id_column}")
        if id_column not in answers_cols:
            raise ValueError(f"В файле ответов отсутствует колонка: {id_column}")
        if predict_column not in predict_cols:
            raise ValueError(f"В файле предсказаний отсутствует колонка: {predict_column}")
        if answers_column not in answers_cols:
            raise ValueError(f"В файле ответов отсутствует колонка: {answers_column}")
        
        # Объединяем данные
        merged = (predictions
                  .select(id_column, predict_column)
                  .join(answers.select(id_column, answers_column),
                        on=id_column,
                        how="inner")
                  )
        
        # Определяем условие для поиска ошибок и находим ошибки
        if target_class in [0, 1]:
            # реальный ответ = target class, предсказание иное
            error_condition = (col(answers_column) == target_class) & (col(predict_column) != target_class)
        else:
            raise ValueError("target_class должен быть 0, 1")
        
        errors = merged.filter(error_condition)
        
        # Считаем статистику
        total_count = merged.count()
        mismatch_count = merged.filter(col(predict_column) != col(answers_column)).count()
        error_count = errors.count()
        
        print(f"Всего записей: {total_count}")
        print(f"Всего несоответствий: {mismatch_count}")
        print(f"Найдено целевых ошибок: {error_count}")
        
        if error_count > 0:
            print(f"\nПервые 10 {id_column} с ошибками:")
            errors.select(id_column).show(10)

        return [row[id_column] for row in errors.select(id_column).collect()]
        
    finally:
        spark.stop()
    

In [ ]:
# Поиск лучших параметров и отбор полезных признаков (shap?)

In [ ]:
target_class_mistakes_finder(predict_path=valid_predict_saving_path,
                             answers_path=valid_predict_data_path)

## Test Predict (подготовка решения)

In [ ]:
train_files = [valid_train_data_path, valid_predict_data_path]

train_in_chunks(files_list=train_files,
                cols_json_file='../datasets/joined/columns_list.json',
                model=catboost_model)

predict_in_chunks(file='../datasets/test/test.parquet',
                  cols_json_file='../datasets/joined/columns_list.json',
                  model=catboost_model,
                  save_path='../submits/submit.parquet')

## Submit Creating (Создание сабмита)

In [ ]:
save_submit_path = '../submits/submit.csv'

def create_submit(save_path: str = save_submit_path,
                  take_path: str = '../submits/submit.parquet'):
    result_df = pd.read_parquet(take_path)
    
    row_count = len(result_df)
    if row_count != 633683:
        raise ValueError(f"Представленный датафрейм имеет иное количество строк: {row_count}. Такой сабмит не пройдет. Необходимо 633683 строк")
    
    if list(result_df.columns) != ['event_id', 'predict']:
        raise ValueError(f'Неверные названия столбиков: {result_df.columns}')
    
    with open(save_path, 'w') as file:
        result_df.to_csv(file, sep=',', index=False)

In [ ]:
create_submit()